In [1]:
!pip install pymatgen matminer catboost

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.6/55.6 kB 4.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 825.8/825.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 89.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.6/62.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 104.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 56.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.8/354.8 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.3/332.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━

In [2]:
!pip freeze > requirements_colab.txt

In [ ]:
!python --version

Python 3.13.16


In [ ]:
import pandas as pd
from pymatgen.core import Composition
from matminer.featurizers.composition import (
  ElementProperty,
  Stoichiometry,
  ValenceOrbital,
  BandCenter
)
import json
import numpy as np
# 1. Load dataset
df = pd.read_csv('SuperCon_MTG.csv')

# 2. Define robust parser
def safe_composition(formula):
    try:
        return Composition(formula)
    except Exception:
        return None

# 3. Apply parsing and filter out invalid rows
df['composition'] = df['Chemical_Formula'].apply(safe_composition)

# Count and report removed formulas
invalid_count = df['composition'].isna().sum()
print(f"Removed {invalid_count} invalid chemical formula(s).")

# Keep only valid rows
df = df.dropna(subset=['composition']).reset_index(drop=True)

# 4. Proceed with feature extraction
magpie = ElementProperty.from_preset("magpie")
df = magpie.featurize_dataframe(df, col_id='composition')

stoich = Stoichiometry()
valence = ValenceOrbital()
band = BandCenter()

print("Extracting stoichiometry & symmetry features...")
df = stoich.featurize_dataframe(df, col_id='composition')

print("Extracting valence orbital proportions...")
df = valence.featurize_dataframe(df, col_id='composition')

print("Extracting energy band centers...")
df = band.featurize_dataframe(df, col_id='composition')

# 5. Clean up temporary composition column
df = df.drop(columns=['composition'])

# Save the featurized dataset

print(f"Dataset successfully featurized! Output shape: {df.shape}")

df.to_csv('clean_supercon.csv', index=False)



# 1. Cargar datos
df = pd.read_csv("clean_supercon.csv")
print(f"Filas originales: {df.shape}")
# 2. Eliminar identificadores que causan Data Leakage
df_ml = df.drop(columns=['Material-ID', 'Chemical_Formula'])

# 3. Resolver colisiones (filas con mismas características pero distinta Temp_critical)
# Agrupamos por todas las características numéricas y promediamos la temperatura crítica
features = [c for c in df_ml.columns if c != 'Temp_critical']
df_ml_grouped = df_ml.groupby(features, as_index=False)['Temp_critical'].mean()

# 4. Eliminar Multicolinealidad (Variables redundantes con correlación > 0.95)
corr_matrix = df_ml_grouped.drop(columns=['Temp_critical']).corr().abs()
# Triángulo superior de la matriz de correlación
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
# Identificar columnas a descartar
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

df_final = df_ml_grouped.drop(columns=to_drop)

# 5. Guardar el dataset limpio y listo para Machine Learning
#df_final.to_csv("optimized_supercon.csv", index=False)

feature_cols = [
    c for c in df.columns
    if c != "Temp_critical"
]

with open("feature_columns.json", "w") as f:
    json.dump(feature_cols, f, indent=2)

print(f"Filas originales: {df.shape[0]}")
print(f"Filas tras resolver ruido: {df_ml_grouped.shape[0]}")
print(f"Columnas eliminadas (alta correlación): {len(to_drop)}")
print(f"Shape final del dataset: {df_final.shape}")

Removed 2 invalid chemical formula(s).


ElementProperty:   0%|          | 0/13729 [00:00<?, ?it/s]

Extracting stoichiometry & symmetry features...


Stoichiometry:   0%|          | 0/13729 [00:00<?, ?it/s]

Extracting valence orbital proportions...


...

Extracting energy band centers...


BandCenter:   0%|          | 0/13729 [00:00<?, ?it/s]

Dataset successfully featurized! Output shape: (13729, 150)
Filas originales: (13729, 150)


/tmp/ipykernel_902/2617156812.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ml_grouped = df_ml.groupby(features, as_index=False)['Temp_critical'].mean()
/tmp/ipykernel_902/2617156812.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_ml_grouped = df_ml.groupby(features, as_index=False)['Temp_critical'].mean()
/tmp/ipykernel_902/2617156812.py:68: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all col

Filas originales: 13729
Filas tras resolver ruido: 13719
Columnas eliminadas (alta correlación): 28
Shape final del dataset: (13719, 120)


In [ ]:

import joblib
import pandas as pd

from pymatgen.core import Composition
from matminer.featurizers.composition import (
    ElementProperty,
    Stoichiometry,
    BandCenter,
    ValenceOrbital,
)

# --------------------------------------------------
# 1. Compuesto de prueba
# --------------------------------------------------
formula = "CsV3Sb5"
composition = Composition(formula)

# --------------------------------------------------
# 2. Generar las features con los mismos featurizers
# --------------------------------------------------
data = pd.DataFrame({"composition": [composition]})

featurizers = [
    ElementProperty.from_preset("magpie"),
    Stoichiometry(),
    BandCenter(),
    ValenceOrbital(),
]

for featurizer in featurizers:
    data = featurizer.featurize_dataframe(
        data,
        col_id="composition",
        ignore_errors=False,
    )

X_new = data.drop(columns=["composition"])

X_final = X_new.drop(columns=to_drop)


ElementProperty:   0%|          | 0/1 [00:00<?, ?it/s]

Stoichiometry:   0%|          | 0/1 [00:00<?, ?it/s]

BandCenter:   0%|          | 0/1 [00:00<?, ?it/s]

ValenceOrbital:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:

print(X_final.shape)
print(df_final.shape)

(1, 119)
(13719, 120)


In [ ]:


import joblib
import pandas as pd

from pymatgen.core import Composition
from matminer.featurizers.composition import (
    ElementProperty,
    Stoichiometry,
    BandCenter,
    ValenceOrbital,
)

# --------------------------------------------------
# 1. Compuesto de prueba
# --------------------------------------------------
formula = "CsV3Sb5"
composition = Composition(formula)

# --------------------------------------------------
# 2. Generar las features con los mismos featurizers
# --------------------------------------------------
data = pd.DataFrame({"composition": [composition]})

featurizers = [
    ElementProperty.from_preset("magpie"),
    Stoichiometry(),
    BandCenter(),
    ValenceOrbital(),
]

for featurizer in featurizers:
    data = featurizer.featurize_dataframe(
        data,
        col_id="composition",
        ignore_errors=False,
    )

X_new = data.drop(columns=["composition"])

X_final = X_new.drop(columns=to_drop)


# --------------------------------------------------
# 3. Alinear las features con las del entrenamiento
# --------------------------------------------------
# Se usa el CSV como plantilla de columnas.
# No se utiliza Temp_critical para generar la predicción.
from catboost import CatBoostRegressor
from catboost import CatBoostClassifier
template = pd.read_csv("optimized_supercon.csv", nrows=0)


feature_cols = [
    col for col in template.columns
    if col != "Temp_critical"
]

missing = sorted(set(feature_cols) - set(X_final.columns))
extra = sorted(set(X_final.columns) - set(feature_cols))

if missing or extra:
    raise ValueError(
        f"Las features no coinciden.\n"
        f"Faltantes: {missing}\n"
        f"Adicionales: {extra}\n"
        "Revisa los nombres y versiones de los featurizers."
    )

X_new = X_new.reindex(columns=feature_cols)

if X_new.isna().any().any():
    raise ValueError("Hay valores NaN en las features del compuesto.")

# --------------------------------------------------
# 4. Cargar los modelos YA entrenados
# --------------------------------------------------
catboost_reg = CatBoostRegressor()
catboost_cls = CatBoostClassifier()
catboost_reg.load_model("catboost_regressor.cbm")
catboost_cls.load_model("catboost_classifier.cbm")
models_cls = {
    "LightGBM": joblib.load("lgbm_classifier.pkl"),
    "CatBoost": catboost_cls,
}
models_reg = {
    "LightGBM": joblib.load("lgbm_regressor.pkl"),
    "CatBoost": catboost_reg,
}
# --------------------------------------------------
# 5. Predecir Tc
# --------------------------------------------------
print(f"Compuesto: {formula}")
print("Predicciones de temperatura crítica:\n")
for name_cls, model_cls in models_cls.items():
    # Usamos el objeto model_cls para predecir
    pred_cls = float(model_cls.predict(X_new)[0])

    if pred_cls == 1:
        # Si es superconductor, predecimos Tc con el regresor correspondiente
        model_reg = models_reg[name_cls]
        pred_reg = float(model_reg.predict(X_new)[0])
        print(f"[{name_cls}] {formula} es superconductor. Tc estimada: {pred_reg:.2f} K")
    else:
        print(f"[{name_cls}] {formula} NO es superconductor.")

Compuesto: CsV3Sb5
Predicciones de temperatura crítica:

[LightGBM] CsV3Sb5 es superconductor. Tc estimada: 6.87 K
[CatBoost] CsV3Sb5 NO es superconductor.


/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [ ]:
import joblib
import pandas as pd
from catboost import CatBoostClassifier, CatBoostRegressor
from matminer.featurizers.composition import (
    BandCenter,
    ElementProperty,
    Stoichiometry,
    ValenceOrbital,
)
from pymatgen.core import Composition

# --------------------------------------------------
# 1. Compuestos a evaluar y sus valores reales de literatura (Tc_real)
# --------------------------------------------------
data_compuestos = [
    {"formula": "CsV3Sb5", "Tc_real": 2.50},
    {"formula": "KV3Sb5", "Tc_real": 1.00},
    {"formula": "RbV3Sb5", "Tc_real": 0.92},
    {"formula": "Nd0.8Sr0.2NiO2", "Tc_real": 15.00},
    {"formula": "Pr0.8Sr0.2NiO2", "Tc_real": 12.00},
    {"formula": "UTe2", "Tc_real": 1.60},
    {"formula": "URu2Si2", "Tc_real": 1.50},
    {"formula": "C14H8BrCuN3S8", "Tc_real": 11.60},
]

formulas = [item["formula"] for item in data_compuestos]
compositions = [Composition(f) for f in formulas]

# --------------------------------------------------
# 2. Generar las features con matminer
# --------------------------------------------------
print(f"Featurizando {len(formulas)} compuestos...")
df_mat = pd.DataFrame({"composition": compositions})

featurizers = [
    ElementProperty.from_preset("magpie"),
    Stoichiometry(),
    BandCenter(),
    ValenceOrbital(),
]

for featurizer in featurizers:
    df_mat = featurizer.featurize_dataframe(
        df_mat,
        col_id="composition",
        ignore_errors=False,
    )

X_generated = df_mat.drop(columns=["composition"])

# --------------------------------------------------
# 3. Alinear las features con la plantilla de entrenamiento
# --------------------------------------------------
template = pd.read_csv("optimized_supercon.csv", nrows=0)
feature_cols = [col for col in template.columns if col != "Temp_critical"]

if "to_drop" in locals() or "to_drop" in globals():
    X_generated = X_generated.drop(
        columns=[c for c in to_drop if c in X_generated.columns],
        errors="ignore",
    )

missing = sorted(set(feature_cols) - set(X_generated.columns))
if missing:
    raise ValueError(f"Faltan features requeridas: {missing}")

X_final = X_generated.reindex(columns=feature_cols)

if X_final.isna().any().any():
    raise ValueError("Se detectaron valores NaN en las features.")

# --------------------------------------------------
# 4. Cargar los modelos entrenados
# --------------------------------------------------
catboost_reg = CatBoostRegressor()
catboost_cls = CatBoostClassifier()
catboost_reg.load_model("catboost_regressor.cbm")
catboost_cls.load_model("catboost_classifier.cbm")

models_cls = {
    "LightGBM": joblib.load("lgbm_classifier.pkl"),
    "CatBoost": catboost_cls,
}
models_reg = {
    "LightGBM": joblib.load("lgbm_regressor.pkl"),
    "CatBoost": catboost_reg,
}

# --------------------------------------------------
# 5. Generar predicciones y estructurar DataFrame de resultados
# --------------------------------------------------
results_df = pd.DataFrame(data_compuestos)

for name_model in models_cls.keys():
    model_cls = models_cls[name_model]
    model_reg = models_reg[name_model]

    preds_cls = model_cls.predict(X_final)
    preds_reg = model_reg.predict(X_final)

    # Formatear salidas por si vienen en vectores 1D o 2D
    cls_clean = [
        int(p[0]) if hasattr(p, "__len__") else int(p) for p in preds_cls
    ]
    reg_clean = [
        float(p[0]) if hasattr(p, "__len__") else float(p) for p in preds_reg
    ]

    # Agregar columnas al DataFrame
    results_df[f"Es_Superconductor_{name_model}"] = [
        "Sí" if c == 1 else "No" for c in cls_clean
    ]
    results_df[f"Tc_Pred_{name_model}"] = [round(r, 2) for r in reg_clean]

# --------------------------------------------------
# 6. Guardar la tabla en un archivo CSV y mostrarla en consola
# --------------------------------------------------
output_filename = "comparativa_superconductores.csv"
results_df.to_csv(output_filename, index=False)

print("\n" + "=" * 80)
print(f"TABLA COMPARATIVA GUARDADA EXITOSAMENTE EN '{output_filename}'")
print("=" * 80)
print(results_df.to_string(index=False))

Featurizando 8 compuestos...


ElementProperty:   0%|          | 0/8 [00:00<?, ?it/s]

Stoichiometry:   0%|          | 0/8 [00:00<?, ?it/s]

BandCenter:   0%|          | 0/8 [00:00<?, ?it/s]

ValenceOrbital:   0%|          | 0/8 [00:00<?, ?it/s]


TABLA COMPARATIVA GUARDADA EXITOSAMENTE EN 'comparativa_superconductores.csv'
       formula  Tc_real Es_Superconductor_LightGBM  Tc_Pred_LightGBM Es_Superconductor_CatBoost  Tc_Pred_CatBoost
       CsV3Sb5     2.50                         Sí              6.87                         No              9.61
        KV3Sb5     1.00                         Sí              5.71                         No              9.96
       RbV3Sb5     0.92                         Sí              7.12                         No              8.75
Nd0.8Sr0.2NiO2    15.00                         No             24.22                         No             21.68
Pr0.8Sr0.2NiO2    12.00                         No             23.54                         No             24.05
          UTe2     1.60                         Sí              2.15                         Sí              2.70
       URu2Si2     1.50                         Sí              0.80                         Sí              0.91
 C14H8BrC

/usr/local/lib/python3.13/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
